## **PROJET TINDER - EDA**

### **2. DATA CLEANING**

- Input  : `outputs/data/df_speed_dating_prep.csv`  (produit par le Bloc 1)
- Output : `outputs/data/df_speed_dating_cleaned.csv` 

**Objectif**:
- Normalisation T1 
- Traitement des NaN restants sur career_c
- Détection et traitement des outliers résiduels
- Rapport qualité visuel

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path
from plotly.subplots import make_subplots
 
pio.templates.default = "plotly_white"
 
PALETTE = {
    "match":   "#1D9E75",
    "no_match":"#E24B4A",
    "male":    "#378ADD",
    "female":  "#D4537E",
    "neutral": "#888780",
    "purple":  "#7F77DD",
    "amber":   "#EF9F27",
    "ok":      "#1D9E75",
    "warn":    "#EF9F27",
    "bad":     "#E24B4A",
}
 
PREF_COLS   = ["attr1_1","sinc1_1","intel1_1","fun1_1","amb1_1","shar1_1"]
SCORE_SELF  = ["attr","sinc","intel","fun","amb","shar","like","prob"]
SCORE_OTHER = ["attr_o","sinc_o","intel_o","fun_o","amb_o","shar_o","like_o","prob_o"]
COLORS      = ["#7F77DD","#1D9E75","#378ADD","#EF9F27","#D85A30","#888780","#D4537E","#E24B4A"]
PROJECT_ROOT = Path('../')
INPUT_DATA_PATH = PROJECT_ROOT / 'outputs' / 'data' / 'df_speed_dating_prep.csv'
OUTPUT_DATA_PATH = PROJECT_ROOT / 'outputs' / 'data' / 'df_speed_dating_cleaned.csv'

print("=" * 65)
print("DATA CLEANING - NORMALISATION")
print(INPUT_DATA_PATH, OUTPUT_DATA_PATH)
print("=" * 65)

DATA CLEANING - NORMALISATION
../outputs/data/df_speed_dating_prep.csv ../outputs/data/df_speed_dating_cleaned.csv


In [8]:
# =============================================================================
# CHARGEMENT DU DATASET
# =============================================================================
df = pd.read_csv(INPUT_DATA_PATH)
print(f"\nDATASET : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
nan_before = df.isnull().sum()
print(nan_before)


DATASET : 8,378 lignes × 48 colonnes
iid                0
id                 1
gender             0
wave               0
condtn             0
match              0
dec                0
attr             202
sinc             277
intel            296
fun              350
amb              712
shar            1067
like             240
prob             309
dec_o              0
attr_o           212
sinc_o           287
intel_o          306
fun_o            360
amb_o            722
shar_o          1076
like_o           250
prob_o           318
attr1_1           79
sinc1_1           79
intel1_1          79
fun1_1            89
amb1_1            99
shar1_1          121
attr3_1          105
sinc3_1          105
fun3_1           105
intel3_1         105
amb3_1           105
int_corr         158
samerace           0
age_o            104
race_o            73
age               95
race              63
field_cd          82
goal              79
go_out            79
career_c         138
income          4

In [9]:
# =============================================================================
# NORMALISATION T1
# =============================================================================
print("\n" + "─" * 65)
print("NORMALISATION PRÉFÉRENCES T1")
print("─" * 65)

# Le Bloc 1 a ÷10 les waves 100pts (1-5, 10-21).
# Les waves 6-9 stockent aussi leurs scores en base 100
# (ex: score 1-10 encodé en pourcentage → 16.67, 12.77...).
# On normalise toutes les lignes où au moins une pref > 10.
 
mask = df[PREF_COLS].max(axis=1) > 10
n_to_norm = mask.sum()
 
# Sauvegarder attr1_1 brut pour la visualisation avant/après
attr_brut = df["attr1_1"].copy()
attr_brut[mask] = attr_brut[mask] * 10   # reconstruire les valeurs originales
 
print(f"\nLignes non encore normalisées (pref > 10) : {n_to_norm:,}")
print(f"Waves concernées : {sorted(df.loc[mask, 'wave'].unique())}")
print(f"\nMoyennes avant ÷10 : {df.loc[mask, PREF_COLS].mean().round(2).to_dict()}")
 
df.loc[mask, PREF_COLS] = df.loc[mask, PREF_COLS] / 10
 
# Clip résiduel (erreurs de saisie rares)
for col in PREF_COLS:
    df[col] = df[col].clip(upper=10)
 
print(f"Moyennes après ÷10 : {df[PREF_COLS].mean().round(2).to_dict()}")
print(f"Max attr1_1 après  : {df['attr1_1'].max():.2f}")
print(f"Toutes les préférences T1 sur échelle 0-10")
 
# Visualisation avant / après
fig_norm = make_subplots(rows=1, cols=2,
    subplot_titles=["attr1_1 — Avant normalisation","attr1_1 — Après normalisation"])
fig_norm.add_trace(go.Histogram(x=attr_brut, nbinsx=30,
    marker_color=PALETTE["bad"], opacity=0.75, showlegend=False), row=1, col=1)
fig_norm.add_trace(go.Histogram(x=df["attr1_1"], nbinsx=30,
    marker_color=PALETTE["ok"], opacity=0.75, showlegend=False), row=1, col=2)
fig_norm.update_layout(
    title="P1 — Normalisation préférences T1 (attr1_1 = importance attractivité déclarée)",
    height=360, plot_bgcolor="white", title_font_size=15)
fig_norm.update_xaxes(title_text="Valeur brute", row=1, col=1)
fig_norm.update_xaxes(title_text="Valeur normalisée (0-10)", row=1, col=2)
fig_norm.show()


─────────────────────────────────────────────────────────────────
NORMALISATION PRÉFÉRENCES T1
─────────────────────────────────────────────────────────────────

Lignes non encore normalisées (pref > 10) : 1,557
Waves concernées : [np.int64(6), np.int64(7), np.int64(8), np.int64(9)]

Moyennes avant ÷10 : {'attr1_1': 16.16, 'sinc1_1': 17.82, 'intel1_1': 18.99, 'fun1_1': 17.91, 'amb1_1': 14.73, 'shar1_1': 14.39}
Moyennes après ÷10 : {'attr1_1': 2.25, 'sinc1_1': 1.74, 'intel1_1': 2.03, 'fun1_1': 1.75, 'amb1_1': 1.07, 'shar1_1': 1.18}
Max attr1_1 après  : 10.00
Toutes les préférences T1 sur échelle 0-10


In [10]:
# =============================================================================
# NaN RÉSIDUELS : career_c (138 lignes)
# =============================================================================
print("\n" + "─" * 65)
print("NaN RÉSIDUELS : career_c")
print("─" * 65)
 
n_nan_career = df["career_c"].isnull().sum()
print(f"\ncareer_c NaN : {n_nan_career} ({n_nan_career/len(df)*100:.1f}%)")
 
mode_career = int(df["career_c"].mode()[0])
df["career_c"] = df["career_c"].fillna(mode_career).astype(int)
 
career_map = {
    1:"Lawyer", 2:"Academic", 3:"Psychologist", 4:"Doctor",
    5:"Engineer", 6:"Creative Arts", 7:"Finance/Business", 8:"Real Estate",
    9:"Intl Affairs", 10:"Undecided", 11:"Social Work", 12:"Speech Pathology",
    13:"Politics", 14:"Pro Sports", 15:"Other", 16:"Journalism", 17:"Architecture"
}
print(f"Imputation par mode : {mode_career} -> {career_map.get(mode_career,'?')}")
print(f"career_c NaN restants : {df['career_c'].isnull().sum()}")
 
print(f"\nincome : {df['income'].isnull().sum():,} NaN conservés intentionnellement")
print(f"> Participants étrangers sans code postal US (biais géographique)")


─────────────────────────────────────────────────────────────────
NaN RÉSIDUELS : career_c
─────────────────────────────────────────────────────────────────

career_c NaN : 138 (1.6%)
Imputation par mode : 2 -> Academic
career_c NaN restants : 0

income : 4,099 NaN conservés intentionnellement
> Participants étrangers sans code postal US (biais géographique)


In [13]:
# =============================================================================
# OUTLIERS RÉSIDUELS
# =============================================================================
print("\n" + "─" * 65)
print("VÉRIFICATION OUTLIERS RÉSIDUELS")
print("─" * 65)
 
# Scores soirée hors [0, 10]
print("\nScores soirée hors [0, 10] :")
n_fixed = 0
for col in SCORE_SELF + SCORE_OTHER:
    if col in df.columns:
        n_out = ((df[col] < 0) | (df[col] > 10)).sum()
        if n_out > 0:
            df[col] = df[col].clip(0, 10)
            n_fixed += n_out
            print(f"x {col} : {n_out} valeurs clippées")
if n_fixed == 0:
    print("Aucune valeur hors [0,10] détectée")
 
# Préférences T1 hors [0, 10] après normalisation
print("\n  Préférences T1 hors [0, 10] après normalisation :")
n_fixed_pref = 0
for col in PREF_COLS:
    n_out = ((df[col] < 0) | (df[col] > 10)).sum()
    if n_out > 0:
        df[col] = df[col].clip(0, 10)
        n_fixed_pref += n_out
        print(f"x {col} : {n_out} valeurs clippées")
if n_fixed_pref == 0:
    print("Aucune valeur hors [0,10] détectée")
 
# Âge
print(f"\nÂge — plage : {df['age'].min():.0f} – {df['age'].max():.0f} ans")
print(f"age > 50    : {(df['age'] > 50).sum()} lignes => légitimes, conservées")
 
# Boxplots
fig_box = make_subplots(rows=1, cols=2,
    subplot_titles=["Scores sujet → partenaire", "Scores partenaire / sujet"])
for i, col in enumerate(SCORE_SELF):
    fig_box.add_trace(go.Box(y=df[col], name=col,
        marker_color=COLORS[i], boxmean=True, showlegend=False), row=1, col=1)
for i, col in enumerate(SCORE_OTHER):
    fig_box.add_trace(go.Box(y=df[col], name=col,
        marker_color=COLORS[i], boxmean=True, showlegend=False), row=1, col=2)
fig_box.update_layout(
    title="P3 — Distributions des scores soirée (après cleaning complet)",
    height=420, plot_bgcolor="white", title_font_size=15)
fig_box.update_yaxes(title_text="Score (0-10)")
fig_box.show()


─────────────────────────────────────────────────────────────────
VÉRIFICATION OUTLIERS RÉSIDUELS
─────────────────────────────────────────────────────────────────

Scores soirée hors [0, 10] :
Aucune valeur hors [0,10] détectée

  Préférences T1 hors [0, 10] après normalisation :
Aucune valeur hors [0,10] détectée

Âge — plage : 18 – 55 ans
age > 50    : 6 lignes => légitimes, conservées


In [15]:
# =============================================================================
# RAPPORT QUALITÉ VISUEL
# =============================================================================
print("\n" + "─" * 65)
print("RAPPORT QUALITÉ FINAL")
print("─" * 65)
 
nan_after = df.isnull().sum()
nan_rpt = pd.DataFrame({"Avant cleaning": nan_before, "Apres cleaning": nan_after})
nan_rpt = nan_rpt[(nan_rpt["Avant cleaning"] > 0) | (nan_rpt["Apres cleaning"] > 0)].copy()
nan_rpt["Réduction"] = nan_rpt["Avant cleaning"] - nan_rpt["Apres cleaning"]
nan_rpt["% restant"] = (nan_rpt["Apres cleaning"] / len(df) * 100).round(1)
 
print("\nBilan NaN avant - après cleaning :")
print(nan_rpt.to_string())
 
analytic_cols = SCORE_SELF + SCORE_OTHER + PREF_COLS + ["age","race","samerace","int_corr","goal"]
nan_analytic = df[[c for c in analytic_cols if c in df.columns]].isnull().sum().sum()
print(f"\nNaN colonnes analytiques  : {nan_analytic}")
print(f"NaN income (intentionnel) : {df['income'].isnull().sum():,}")
print(f"NaN total dataset         : {df.isnull().sum().sum():,}")
 
# Dashboard 4 graphiques
fig_dash = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "NaN restants par colonne (% du dataset)",
        "Préférences T1 — violin après normalisation",
        "Scores soirée — moyenne par attribut",
        "career_c — répartition après imputation",
    ],
    vertical_spacing=0.18, horizontal_spacing=0.10,
)
 
# 1. NaN restants
nan_plot = nan_after[nan_after > 0].sort_values(ascending=False)
pct_plot = (nan_plot / len(df) * 100).round(1)
fig_dash.add_trace(go.Bar(
    x=nan_plot.index, y=pct_plot.values, showlegend=False,
    marker_color=[PALETTE["bad"] if v > 30 else PALETTE["warn"] if v > 5
                  else PALETTE["ok"] for v in pct_plot.values],
    text=pct_plot.astype(str) + "%", textposition="outside",
), row=1, col=1)
 
# 2. Violin préférences T1
pref_short = {"attr1_1":"Attr","sinc1_1":"Sinc","intel1_1":"Intel",
              "fun1_1":"Fun","amb1_1":"Amb","shar1_1":"Shar"}
for col, clr in zip(PREF_COLS, COLORS):
    fig_dash.add_trace(go.Violin(
        y=df[col], name=pref_short.get(col, col),
        marker_color=clr, box_visible=True,
        meanline_visible=True, showlegend=False, opacity=0.8,
    ), row=1, col=2)
 
# 3. Moyennes scores soirée
score_means = df[SCORE_SELF].mean().round(2).reset_index()
score_means.columns = ["attr","mean"]
score_means["attr_up"] = score_means["attr"].str.upper()
fig_dash.add_trace(go.Bar(
    x=score_means["attr_up"], y=score_means["mean"],
    marker_color=COLORS, showlegend=False,
    text=score_means["mean"], textposition="outside",
), row=2, col=1)
 
# 4. Career_c
career_cnt = df["career_c"].value_counts().sort_index().reset_index()
career_cnt.columns = ["code","count"]
career_cnt["label"] = career_cnt["code"].map(career_map).fillna("Other")
fig_dash.add_trace(go.Bar(
    x=career_cnt["label"], y=career_cnt["count"],
    marker_color=PALETTE["purple"], showlegend=False,
), row=2, col=2)
 
fig_dash.update_layout(
    title="Dashboard qualité — cleaning Data",
    height=640, plot_bgcolor="white", title_font_size=15,
)
fig_dash.update_xaxes(tickangle=-45, row=1, col=1)
fig_dash.update_xaxes(tickangle=-45, row=2, col=2)
fig_dash.update_yaxes(title_text="% NaN", row=1, col=1)
fig_dash.update_yaxes(title_text="Score (0-10)", row=1, col=2)
fig_dash.update_yaxes(title_text="Moyenne (0-10)", row=2, col=1)
fig_dash.update_yaxes(title_text="Nb lignes", row=2, col=2)
fig_dash.show()


─────────────────────────────────────────────────────────────────
RAPPORT QUALITÉ FINAL
─────────────────────────────────────────────────────────────────

Bilan NaN avant - après cleaning :
          Avant cleaning  Apres cleaning  Réduction  % restant
id                     1               1          0        0.0
attr                 202             202          0        2.4
sinc                 277             277          0        3.3
intel                296             296          0        3.5
fun                  350             350          0        4.2
amb                  712             712          0        8.5
shar                1067            1067          0       12.7
like                 240             240          0        2.9
prob                 309             309          0        3.7
attr_o               212             212          0        2.5
sinc_o               287             287          0        3.4
intel_o              306             306          0  

In [16]:
# =============================================================================
# EXPORT
# =============================================================================
df.to_csv(OUTPUT_DATA_PATH, index=False)
 
print(f"\n{'=' * 65}")
print("BLOC 2 TERMINÉ")
print(f"{'=' * 65}")


BLOC 2 TERMINÉ
